# Initial data import

Initial data import from Bigquery to local csv files. This is a one-time process, and the resulting csv files will be used for all subsequent analysis and plotting.

In [3]:
import folium
import geopandas as gp
import pandas as pd

In [12]:
df = pd.read_csv("/workspaces/capitalbikeshare_sql/tripdata.csv")
df.head()

,weekday,weekday_order,member_casual,start_lat,start_lng,trip_count
0,Sunday,1,member,38.915544,-77.038252,148
1,Sunday,1,member,38.917764,-77.032096,143
2,Sunday,1,member,38.909801,-77.034427,125
3,Sunday,1,member,38.910000,-77.030000,118
4,Sunday,1,member,38.913046,-77.032008,112


In [ ]:
from folium.plugins import HeatMap

# Interactive heatmap of the number of trips starting at each station by weekday

# Create a GeoDataFrame from the station data
stations = pd.read_csv("/workspaces/capitalbikeshare_sql/tripdata.csv")
gdf = gp.GeoDataFrame(
    stations, geometry=gp.points_from_xy(stations.start_lng, stations.start_lat)
)
gdf = gdf.set_crs("EPSG:4326")

m = folium.Map(location=[gdf.geometry.y.mean(), gdf.geometry.x.mean()], zoom_start=12)

days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

for day in days:
    day_data = gdf[gdf["weekday"] == day][
        ["start_lat", "start_lng", "trip_count"]
    ].values.tolist()
    fg = folium.FeatureGroup(name=day, show=False)
    HeatMap(data=day_data, radius=15, blur=10).add_to(fg)
    fg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m.save("map.html")